# Module 307 — Code Injection and Safe Command Handling

This worksheet models a narrow feature: validate an IP address and construct the arguments for a fixed `ping` operation. It does not run the command, use a shell, contact a host, or reproduce the HTB exploit.

**Core boundary:** an LLM response is text. Passing that text to a shell asks a powerful interpreter to turn it into actions. Prefer application-selected operations and validated arguments.

## 1. Validate the input as an IP address

`ipaddress.ip_address` parses the entire value as IPv4 or IPv6. Shell punctuation, spaces, and command fragments are not valid IP addresses, so they fail validation. This is type validation: accept an IP value, not an arbitrary string that merely resembles a hostname. A production feature may also need an explicit destination allowlist to prevent access to sensitive internal services.

In [ ]:
import ipaddress

def build_ping_arguments(host_text):
    # Parse the full value; reject anything that is not a bare IP address.
    address = ipaddress.ip_address(host_text)

    # The operation and its options are selected by application code.
    # Return separate arguments; do not join them into a shell command.
    return ['ping', '-c', '3', str(address)]

build_ping_arguments('127.0.0.1')

## 2. Reject values outside the feature's input type

A value containing extra shell syntax is not an IP address, so parsing raises `ValueError` before any process is started. In real code, catch this error and return a safe validation message. Do not try to strip a few suspicious characters and then execute the remainder; define the accepted input type and reject values that do not match it.

In [ ]:
try:
    build_ping_arguments('127.0.0.1; extra text')
except ValueError as error:
    print('Rejected before command execution:', error)

## 3. Why argument separation matters

A real implementation would pass the returned list to a process API with shell interpretation disabled, plus a timeout and suitable output limits. The operating system then receives an executable and distinct arguments; characters inside an argument are not re-parsed as shell operators. This worksheet intentionally stops before launching a process.

A stronger design still checks authorization: even a syntactically valid IP could point somewhere the feature should not reach. Use a destination allowlist, least-privilege process identity, and network isolation as appropriate. Do not let the model choose an arbitrary executable or operation.

## COAE quick review

- **Risk:** untrusted model output reaches a shell or command runner, creating server-side command execution.
- **Why a prompt/filter is weak:** the model may generate unexpected syntax, and a string filter may miss alternate forms or operators.
- **Safer pattern:** fixed operation, strict input type, separate argument list, shell disabled, least privilege, timeout, and isolation.
- **Distinguish from SQL injection:** both involve untrusted text crossing into an interpreter, but one targets database query execution and the other operating-system command execution.

No command in this notebook is executed.